<a href="https://colab.research.google.com/github/eceirem/COVID19-Pneumonia-XRay-Classification/blob/main/notebooks/02_baseline_and_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'

# Ensure the mountpoint is clean before attempting to mount.
if os.path.exists(mountpoint):
    # If it exists and is a mount point, unmount it first.
    if os.path.ismount(mountpoint):
        try:
            drive.flush_and_unmount() # Colab-specific function to unmount Drive
            print(f"Unmounted existing drive from {mountpoint}.")
        except Exception as e:
            print(f"Could not unmount existing drive: {e}. Trying to clear anyway.")
    else:
        # If it's just a regular directory with files, remove it.
        try:
            shutil.rmtree(mountpoint)
            print(f"Removed non-empty local directory '{mountpoint}'.")
        except Exception as e:
            print(f"Could not remove local directory '{mountpoint}': {e}. Trying to mount anyway.")

# Recreate the directory to ensure it exists and is empty
os.makedirs(mountpoint, exist_ok=True)
print(f"Ensured mountpoint '{mountpoint}' exists and is empty.")

drive.mount(mountpoint, force_remount=True)

In [ ]:
"""
02_baseline_and_ml.ipynb
Author: Ece
Description: Classical Machine Learning Baselines for the Ablation Study.
Extracts HOG (Histogram of Oriented Gradients) features from the balanced dataset.
Trains Support Vector Machines (SVM), Logistic Regression, and Random Forest models.
Evaluates the models on a 20% unseen Test Set and exports results to an Excel file.
"""

import os
import cv2
import zipfile
import pandas as pd
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive

# --- 1. CONFIGURATION ---
IMG_SIZE = (224, 224)
CLASSES = ["COVID", "Normal", "Viral Pneumonia"]

# KESİN VE SENKRONİZE EDİLMİŞ YOLLAR
DRIVE_MASKED_ZIP = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Dataset/Preprocessed/Maskeli/Balanced_Dataset_Dogukan.zip"

DRIVE_NOT_MASKED_ZIP = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Dataset/Preprocessed/Maskesiz/Balanced_Dataset_Dogukan_NotMasked.zip"

# Local extraction paths for fast I/O
LOCAL_BASE = "/content/local_ml_data"
EXCEL_OUTPUT_PATH = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Results/ML_Ablation_Results.xlsx"

# Make sure results directory exists in Drive
os.makedirs(os.path.dirname(EXCEL_OUTPUT_PATH), exist_ok=True)

def extract_zip(zip_path, extract_to):
    """Extracts dataset to local storage safely."""
    print(f"[INFO] Extracting {os.path.basename(zip_path)}...")
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"[ERROR] Zip file not found at: {zip_path}")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    return extract_to

def find_dataset_root(base_dir):
    """Finds the root directory containing the class folders."""
    for root, dirs, files in os.walk(base_dir):
        if all(c in dirs for c in CLASSES):
            return root
    return base_dir

def load_data_and_extract_features(dataset_path):
    """
    Reads images, converts to Grayscale, resizes to 224x224,
    and extracts HOG features. Returns feature matrix X and labels y.
    """
    X, y = [], []
    root_path = find_dataset_root(dataset_path)
    print(f"[INFO] Loading images and extracting HOG features from: {root_path}")

    for label_idx, cls in enumerate(CLASSES):
        cls_path = os.path.join(root_path, cls)
        img_names = os.listdir(cls_path)

        for img_name in img_names:
            img_path = os.path.join(cls_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                continue

            img = cv2.resize(img, IMG_SIZE)

            # Extract HOG features (pixels per cell and cells per block are optimized for 224x224)
            features = hog(img, orientations=9, pixels_per_cell=(16, 16),
                           cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

            X.append(features)
            y.append(label_idx)

    return np.array(X), np.array(y)

def train_and_evaluate(X_train, X_test, y_train, y_test, dataset_name):
    """Trains 3 baseline models and calculates evaluation metrics."""
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "SVM (RBF Kernel)": SVC(kernel='rbf', random_state=42)
    }

    results = []

    for model_name, model in models.items():
        print(f"       -> Training {model_name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Calculate Metrics (Macro average handles multi-class outputs)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')

        results.append({
            "Dataset Paradigm": dataset_name,
            "Model": model_name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1
        })
        print(f"          [Done] Accuracy: {acc:.4f} | F1: {f1:.4f}")

    return results

# --- MAIN EXECUTION PIPELINE ---
if __name__ == "__main__":
    print("[INFO] Checking Google Drive connection...")

    # Drive daha önce bağlanmamışsa bağla, bağlıysa es geç.
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print("[INFO] Google Drive is already mounted. Proceeding...")

    # Define Ablation experiments
    experiments = {
        "MASKED": DRIVE_MASKED_ZIP,
        "NOT_MASKED": DRIVE_NOT_MASKED_ZIP
    }

    all_results = []

    for data_name, zip_path in experiments.items():
        print(f"\n{'='*50}\n🚀 STARTING ML EXPERIMENT: {data_name} DATASET\n{'='*50}")

        # 1. Clean local dir and Extract
        extract_dir = os.path.join(LOCAL_BASE, data_name)
        os.makedirs(extract_dir, exist_ok=True)
        extract_zip(zip_path, extract_dir)

        # 2. Extract HOG Features
        X, y = load_data_and_extract_features(extract_dir)
        print(f"[INFO] Extracted Features Shape: {X.shape} | Labels Shape: {y.shape}")

        # 3. Split into 80% Train, 20% Unseen Test Set
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        print(f"[INFO] Split Data - Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

        # 4. Train Models and Evaluate
        results = train_and_evaluate(X_train, X_test, y_train, y_test, data_name)
        all_results.extend(results)

    # 5. Export to Excel
    print("\n[INFO] Compiling Results into Pandas DataFrame...")
    df_results = pd.DataFrame(all_results)

    # Save to Drive
    df_results.to_excel(EXCEL_OUTPUT_PATH, index=False)
    print(f"\n[SUCCESS] Pipeline Complete! Results saved as Excel file at:\n {EXCEL_OUTPUT_PATH}")

    # Display the final table in the Colab output
    display(df_results)